In [1]:
from ingest import load_data
documents = load_data()

In [2]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [13]:
data_gen_instructions = """
You emulate a user who is searching for quick recipes based on their time or 
available groceries or nutrition and dietary preferences.
Formulate 2 questions this user might ask to get particular recipe suggested. 
Questions don't need to strictly correspond or to match the recipe by every attribute or any attribute completely.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [14]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [15]:
doc = documents[100]

In [16]:
import json
user_prompt = json.dumps(doc)

In [17]:
doc

{'id': '200',
 'recipe_name': 'Cherry Chicken',
 'total_time': '45 mins',
 'servings': '4',
 'ingredients': '3 tablespoons vegetable oil, 1 (4 pound) whole chicken, cut into 8 pieces,   salt and pepper to taste, ½ cup all-purpose flour for dusting, 1 (15 ounce) can pitted dark cherries packed in water, ½ cup white sugar, 1 tablespoon cornstarch, 1  orange - with peel, quartered and thinly sliced, ½ cup slivered almonds, toasted',
 'directions': 'Heat oil in a large skillet over medium-high heat. Season chicken with salt and pepper, then coat with flour. Fry in hot oil until browned, turning as needed. Reduce heat to medium, cover, and cook until meat is tender and juices run clear, about 25 minutes.\nRemove chicken from the skillet; pour off all but 1/4 cup drippings. Return the skillet to medium heat and stir in cherries, reserving some cherry liquid for later. Stir in sugar and bring to a boil. Dissolve cornstarch in reserved cherry liquid, then stir into the skillet. Cook, stirring 

In [18]:
user_prompt

'{"id": "200", "recipe_name": "Cherry Chicken", "total_time": "45 mins", "servings": "4", "ingredients": "3 tablespoons vegetable oil, 1 (4 pound) whole chicken, cut into 8 pieces,   salt and pepper to taste, \\u00bd cup all-purpose flour for dusting, 1 (15 ounce) can pitted dark cherries packed in water, \\u00bd cup white sugar, 1 tablespoon cornstarch, 1  orange - with peel, quartered and thinly sliced, \\u00bd cup slivered almonds, toasted", "directions": "Heat oil in a large skillet over medium-high heat. Season chicken with salt and pepper, then coat with flour. Fry in hot oil until browned, turning as needed. Reduce heat to medium, cover, and cook until meat is tender and juices run clear, about 25 minutes.\\nRemove chicken from the skillet; pour off all but 1/4 cup drippings. Return the skillet to medium heat and stir in cherries, reserving some cherry liquid for later. Stir in sugar and bring to a boil. Dissolve cornstarch in reserved cherry liquid, then stir into the skillet. 

In [19]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [20]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [21]:
response.output_parsed.questions

['Got any easy chicken recipes with fruit or a sweet-savory sauce?',
 'What’s a decent 45-minute dinner using chicken and cherries?']

In [22]:
from evaluation_utils import llm_structured

In [23]:
from evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [26]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/598 [00:00<?, ?it/s]

In [27]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

1196

In [28]:
ground_truth[10]

{'question': 'I’ve got a bunch of apples and want something quick—what’s an easy warm dessert I can make in about 20 minutes?',
 'document': '14'}

In [30]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.3355124999999997

In [31]:
import pandas as pd

In [32]:
df_ground_truth = pd.DataFrame(ground_truth)

In [33]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)